In [0]:
# %pip install -U --quiet databricks-langchain databricks-agents mlflow databricks-vectorsearch langchain langchain-core PyMuPDF
# dbutils.library.restartPython()


In [0]:
import fitz
from operator import itemgetter
from langchain_text_splitters import RecursiveCharacterTextSplitter
from databricks.vector_search.client import VectorSearchClient
from databricks_langchain import ChatDatabricks, DatabricksVectorSearch
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

In [0]:
# Changing values here updates the entire notebook
chain_config = {
    "volume_path": "/Volumes/db_workspace_claire/default/these/2025-09-eb-big-book-of-machine-learning-3rd-ed.pdf",
    "delta_table_name": "db_workspace_claire.default.chunked_data",
    "vector_search_endpoint": "vector_search_endpoint_name",
    "vector_index_path": "db_workspace_claire.default.big_book_of_machine_learning_index",
    "embedding_endpoint": "bge_base_en_v1_5",
    "llm_endpoint": "databricks-meta-llama-3.1-405b-instruct",
    "llm_prompt_template": """You are an assistant that answers questions based on a specific document. 
    Use the following pieces of retrieved context to answer the question. 
    If you don't know the answer, just say you don't know.
    
    Context: {context}"""
}

In [0]:
def extract_and_chunk(path):
    doc = fitz.open(path)
    text = "".join([page.get_text() for page in doc])
    doc.close()
    
    splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)
    chunks = splitter.split_text(text)
    
    return [{"source": path, "chunk_id": i, "content": chunk} for i, chunk in enumerate(chunks)]

data = extract_and_chunk(chain_config.get("volume_path"))

In [0]:
spark.createDataFrame(data).write.mode("overwrite") \
    .option("delta.enableChangeDataFeed", "true") \
    .saveAsTable(chain_config.get("delta_table_name"))

In [0]:
vsc = VectorSearchClient()
try:
    vsc.get_endpoint(name=chain_config.get("vector_search_endpoint"))
    print(f" Endpoint '{chain_config.get('vector_search_endpoint')}' is already there.")
except:
    print(f"Creating endpoint '{chain_config.get('vector_search_endpoint')}'... This takes ~10 mins.")
    vsc.create_endpoint(name=chain_config.get("vector_search_endpoint"), endpoint_type="STANDARD")

In [0]:
vsc = VectorSearchClient()

# Check if index exists, if not, create it
try:
    index = vsc.get_index(chain_config.get("vector_search_endpoint"), chain_config.get("vector_index_path"))
except:
    index = vsc.create_delta_sync_index(
        endpoint_name=chain_config.get("vector_search_endpoint"),
        source_table_name=chain_config.get("delta_table_name"),
        index_name=chain_config.get("vector_index_path"),
        pipeline_type="TRIGGERED",
        primary_key="chunk_id",
        embedding_source_column="content",
        embedding_model_endpoint_name=chain_config.get("embedding_endpoint")
    )
    

In [0]:
chat_model = ChatDatabricks(endpoint=chain_config.get("llm_endpoint"), temperature=0.1)

vector_store = DatabricksVectorSearch(index_name=chain_config.get("vector_index_path"))
retriever = vector_store.as_retriever(search_kwargs={"k": 3})

# Helper function to turn documents into a single string for the prompt
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


In [0]:
prompt = ChatPromptTemplate.from_messages([
    ("system", chain_config.get("llm_prompt_template")),
    ("user", "{question}")
])

# This chain: Takes question -> Finds Docs -> Formats them -> Sends to LLM -> Cleans string
rag_chain = (
    {
        "context": itemgetter("question") | retriever | format_docs,
        "question": itemgetter("question")
    }
    | prompt
    | chat_model
    | StrOutputParser()
)

In [0]:
question = "What are some real-world applications of machine learning mentioned in the guide?"
print(rag_chain.invoke({"question": question}))